# EDN-Ad: Train on DFDC → Cross-test on CelebDF-v2

**Порядок подготовки перед запуском:**

1. ⚡ Settings → Accelerator → **GPU T4 x2** (используется 1 GPU, вторая отключается для экономии RAM)
2. 📦 Data → Add Data → найти **`deepfake-detection-challenge`** (официальный конкурс Kaggle) → Add
3. 📦 Data → Add Data → найти **`celeb-df-v2`** (community dataset) → Add
4. ▶️ Run All

**Что делает ноутбук:**
- Декодирует FF++ mp4 с **детекцией лица** (Haar cascade + center crop fallback)
- Обучает EDN-Ad: EfficientNet-B0 + FPN + SE → TCN → Linear classifier (BCE loss)
- Backbone freeze первые 2 эпохи → unfreeze с пониженным LR
- Аугментация: flip, color jitter, Gaussian blur, JPEG compression
- Оценивает на **CelebDF-v2** без дообучения (cross-dataset)
- Сохраняет `best_model.pt`, графики, итоговую таблицу

**Ключевые решения (из анализа лучших практик DFDC):**
- BCE loss вместо ArcFace (все топ-решения DFDC: selimsef, NTech-Lab, конкурсанты)
- Face detection для FF++ (как в 1st place — selimsef)
- Backbone freeze → differential LR (pretrained features не разрушаются)
- Сильная аугментация (JPEG, blur, jitter — имитация real-world деградации)

In [ ]:
# ─── 0. Зависимости ────────────────────────────────────────────────────────
# КРИТИЧНО: ограничиваем GPU ДО любых импортов torch.
# DataParallel на 2×T4 жрёт ~25 MB CPU RAM/батч (PyTorch bug #46753).
# Плюс каждый GPU контекст = ~2 GB HOST RAM. Итого 2 GPU = 4 GB host + утечка.
# Одна GPU: 2 GB host, нет утечки, скорость -30% но RAM -10 GB.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # 1 GPU — нет DataParallel, нет утечки

# КРИТИЧНО: настройка glibc malloc ПЕРЕД всеми импортами.
# Проблема: torch.tensor() аллоцирует ~10 MB float32 на каждый сэмпл.
# После free() glibc НЕ возвращает память ОС (хранит в "аренах").
# За 150 батчей RSS растёт на 8-10 GB → OOM.
# Решение: mallopt заставляет glibc использовать mmap для больших аллокаций.
# При free → munmap мгновенно возвращает память ОС.
import ctypes, ctypes.util
_c_lib = ctypes.CDLL(ctypes.util.find_library('c'))
try:
    _c_lib.mallopt(-3, 65536)    # M_MMAP_THRESHOLD = 64 KB
    _c_lib.mallopt(-1, 131072)   # M_TRIM_THRESHOLD = 128 KB
    _c_lib.mallopt(-8, 2)        # M_ARENA_MAX = 2 (вместо 8*cores)
    print('glibc malloc tuned ✓ (mmap >64KB, arena_max=2)')
except Exception as e:
    print(f'glibc malloc tuning skipped: {e}')


def _malloc_trim():
    """Возвращает неиспользуемую heap-память ОС."""
    try: _c_lib.malloc_trim(0)
    except Exception: pass


# ВАЖНО: если было сообщение об ошибке numpy/sklearn — сначала сделай
# Run → Restart Session, затем запускай ячейки заново.
import subprocess, sys, importlib

def can_import(name):
    try: importlib.import_module(name); return True
    except Exception: return False

def run_pip(*args, **kwargs):
    return subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(args),
        capture_output=True, timeout=180
    ).returncode == 0

# Kaggle T4-образ уже содержит: av, timm, sklearn, cv2, scipy, torch
# Устанавливаем только то, чего реально нет, и НЕ трогаем numpy/scipy
missing = [pkg for pkg, mod in [('av','av'),('timm','timm')] if not can_import(mod)]
if missing:
    print(f'Installing missing: {missing}')
    for pkg in missing:
        ok = run_pip(pkg, '--no-deps')
        if not ok:
            run_pip(pkg, 'numpy<2.0')

# Диагностика
import numpy as _np
print(f'numpy {_np.__version__}')
ok_all = True
for pkg, mod in [('av','av'),('timm','timm'),('sklearn','sklearn'),('cv2','cv2')]:
    ok = can_import(mod)
    ok_all = ok_all and ok
    print(f'  {mod}: {"ok ✓" if ok else "MISSING ✗"}')

if not ok_all:
    print('\nEСЛИ sklearn MISSING: Run → Restart Session → Run All')
else:
    print('Dependencies ready ✓')

In [ ]:
# ─── 1. Импорты и конфигурация ─────────────────────────────────────────────
import os, io, json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from PIL import Image
import cv2, av

CFG = {
    # ── Модель (EDN-Ad) ──
    'embed_dim':   512,
    'n_layers':    2,          # TCN блоков (2 достаточно для 16 кадров)
    'kernel':      3,
    'dropout':     0.3,        # было 0.1 → увеличено для регуляризации
    # ── Препроцессинг ──
    'target_frames': 16,
    'image_size':    224,
    # ── Обучение ──
    'batch':      8,
    'grad_accum': 4,           # effective batch = 32
    'lr':         1e-3,        # для head/FPN/SE/TCN (random init)
    'lr_backbone': 2e-5,       # для EfficientNet (pretrained) — в 50x меньше
    'wd':         1e-4,
    'epochs':     15,
    'freeze_backbone_epochs': 2,  # замораживаем backbone на 2 эпохи
    'clip_grad':  5.0,
    'seed':       42,
    'num_workers': 1,
    'use_amp':    True,
    # ── Данные ──
    'ffpp_root':    '/kaggle/input/datasets/ahmedelbanby/faceforensicsplusplus-c23-deepfakebench-structure',
    'dfdc_faces':   '/kaggle/input/datasets/itamargr/dfdc-faces-of-the-train-sample',
    'dfdc_sample':  '/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos',
    'celebdf_root': '/kaggle/input/datasets/reubensuju/celeb-df-v2',
    'max_videos_per_class': 1500,
    'ffpp_max_per_class':   1500,
    'val_ratio':    0.10,
    'celebdf_max':  500,
    'output_dir':   '/kaggle/working',
}

n_gpu  = torch.cuda.device_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | GPUs: {n_gpu}')
if n_gpu:
    print(f'GPU 0: {torch.cuda.get_device_name(0)}')

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)

set_seed(CFG['seed'])
print('Config ready ✓')

In [ ]:
# ─── 2. Построение манифестов (FF++ + DFDC faces) ────────────────────────
import re

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')


def _is_frame_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    for suf in IMG_EXTS:
        if next(p.glob(f'*{suf}'), None) is not None:
            return True
    return False


def _find_video_dirs(root: Path, max_depth=5):
    results = []
    if not root.exists():
        return results
    stack = [(root, 0)]
    while stack:
        p, depth = stack.pop()
        if depth > max_depth:
            continue
        try:
            children = list(p.iterdir())
        except Exception:
            continue
        if _is_frame_dir(p):
            results.append(p)
            continue
        for c in children:
            if c.is_dir():
                stack.append((c, depth + 1))
    return results


def build_ffpp_manifest(root, max_per_class=1500):
    """Датасет ahmedelbanby/faceforensicsplusplus-c23-deepfakebench-structure —
    это сборник: frames/<Src>/<video_id>/*.png + videos/FaceForensics++/…
    Берём: real = DFDC_Original, Original_DFDC; fake = Method_A, Method_B."""
    root = Path(root)
    if not root.exists():
        print(f'FF++ не найден: {root}')
        return []

    print(f'Изучаю FF++: {root}')
    frames_root = root / 'frames'
    records = []

    real_sources = ['DFDC_Original', 'Original_DFDC']
    fake_sources = ['Method_A', 'Method_B']

    if frames_root.exists():
        def _collect(names, label, tag):
            out = []
            for name in names:
                d = frames_root / name
                if not d.exists(): continue
                video_dirs = [p for p in sorted(d.iterdir())
                              if p.is_dir() and _is_frame_dir(p)]
                # если прямых video-папок нет, уходим на уровень глубже
                if not video_dirs:
                    video_dirs = _find_video_dirs(d, max_depth=3)
                print(f'  frames/{name}: {len(video_dirs)} папок с кадрами')
                for vd in video_dirs:
                    out.append({'path': str(vd), 'label': label, 'dataset': 'ffpp',
                                 'manipulation': tag, 'source_type': 'folder'})
            return out

        reals = _collect(real_sources, 0, 'none')
        fakes = _collect(fake_sources, 1, 'ffpp_fake')
    else:
        reals, fakes = [], []

    # Доп. источник: videos/FaceForensics++/ (mp4) — стандартная структура DeepfakeBench
    ffpp_vid = root / 'videos' / 'FaceForensics++'
    if ffpp_vid.exists():
        # original_sequences/youtube/c23/videos/*.mp4 — real
        for cand in [ffpp_vid / 'original_sequences' / 'youtube' / 'c23' / 'videos',
                     ffpp_vid / 'original_sequences' / 'youtube' / 'raw' / 'videos',
                     ffpp_vid / 'original_sequences' / 'actors' / 'c23' / 'videos']:
            if cand.exists():
                for vp in sorted(cand.glob('*.mp4')):
                    reals.append({'path': str(vp), 'label': 0, 'dataset': 'ffpp',
                                   'manipulation': 'youtube', 'source_type': 'video'})
                print(f'  videos/{cand.relative_to(ffpp_vid)}: +mp4 real')
                break
        # manipulated_sequences/<Method>/c23/videos/*.mp4 — fake
        ms = ffpp_vid / 'manipulated_sequences'
        if ms.exists():
            for method_dir in sorted(ms.iterdir()):
                if not method_dir.is_dir(): continue
                for cand in [method_dir / 'c23' / 'videos', method_dir / 'raw' / 'videos']:
                    if cand.exists():
                        for vp in sorted(cand.glob('*.mp4')):
                            fakes.append({'path': str(vp), 'label': 1, 'dataset': 'ffpp',
                                           'manipulation': method_dir.name,
                                           'source_type': 'video'})
                        print(f'  videos/manipulated_sequences/{method_dir.name}: +mp4 fake')
                        break

    if not reals and not fakes:
        # Последний fallback — рекурсивный поиск по всему root
        print('  Явная структура не найдена, рекурсивный поиск...')
        video_dirs = _find_video_dirs(root, max_depth=7)
        print(f'  Найдено {len(video_dirs)} папок с кадрами')
        for vd in video_dirs:
            path_str = str(vd).lower()
            if any(k in path_str for k in ['original', 'real', 'youtube', 'actors']):
                reals.append({'path': str(vd), 'label': 0, 'dataset': 'ffpp',
                               'manipulation': 'none', 'source_type': 'folder'})
            elif any(k in path_str for k in
                     ['method_a', 'method_b', 'deepfakes', 'face2face',
                      'faceswap', 'neuraltextures', 'faceshifter', 'manipulated', 'fake']):
                fakes.append({'path': str(vd), 'label': 1, 'dataset': 'ffpp',
                               'manipulation': 'ffpp_fake', 'source_type': 'folder'})

    random.shuffle(reals); random.shuffle(fakes)
    reals = reals[:max_per_class]
    fakes = fakes[:max_per_class]
    print(f'FF++ ИТОГО: real={len(reals)}, fake={len(fakes)}')
    return reals + fakes


def build_dfdc_faces_manifest(root, max_per_class=1500):
    """itamargr/dfdc-faces-of-the-train-sample — структура:
    train/{real,fake}/… и validation/{real,fake}/…"""
    root = Path(root)
    if not root.exists():
        print(f'DFDC faces не найден: {root}')
        return []

    print(f'Изучаю DFDC-faces: {root}')
    records = []
    for split in ['train', 'validation']:
        for sub, lbl in [('real', 0), ('fake', 1)]:
            d = root / split / sub
            if not d.exists():
                continue
            # Вариант A: поддиректории — каждая = видео
            subdirs = [p for p in d.iterdir() if p.is_dir() and _is_frame_dir(p)]
            if subdirs:
                for vd in subdirs:
                    records.append({'path': str(vd), 'label': lbl,
                                     'dataset': 'dfdc_faces',
                                     'manipulation': 'dfdc_fake' if lbl else 'none',
                                     'source_type': 'folder'})
                print(f'  {split}/{sub}: {len(subdirs)} папок-видео')
                continue
            # Вариант B: плоские файлы — группируем по префиксу имени (video_id)
            files = sorted(p for p in d.iterdir()
                            if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
            if not files:
                continue
            groups = {}
            for f in files:
                name = f.stem
                m = re.match(r'^(.+?)[_\-]\d+$', name)
                prefix = m.group(1) if m else name
                groups.setdefault(prefix, []).append(str(f))
            n_added = 0
            for prefix, flist in groups.items():
                if len(flist) < 2:
                    continue
                records.append({'path': str(d), 'files': sorted(flist),
                                 'label': lbl, 'dataset': 'dfdc_faces',
                                 'manipulation': 'dfdc_fake' if lbl else 'none',
                                 'source_type': 'files'})
                n_added += 1
            print(f'  {split}/{sub}: {len(files)} файлов → {n_added} видео-групп')

    real = [r for r in records if r['label'] == 0]
    fake = [r for r in records if r['label'] == 1]
    random.shuffle(real); random.shuffle(fake)
    real = real[:max_per_class]
    fake = fake[:max_per_class]
    print(f'DFDC faces ИТОГО: real={len(real)}, fake={len(fake)}')
    return real + fake


def build_sample_manifest(root):
    """DFDC train_sample_videos — mp4 + metadata.json."""
    root = Path(root)
    meta = root / 'metadata.json'
    if not meta.exists():
        return []
    data = json.loads(meta.read_text())
    records = []
    for fname, info in data.items():
        vp = root / fname
        if not vp.exists():
            continue
        lbl = 1 if info.get('label') == 'FAKE' else 0
        records.append({'path': str(vp), 'label': lbl, 'dataset': 'dfdc_sample',
                         'manipulation': 'dfdc_fake' if lbl else 'none', 'source_type': 'video'})
    return records


def build_celebdf_manifest(root, max_videos=500):
    root = Path(root)
    if not root.exists():
        print(f'CelebDF-v2 не найден: {root}')
        return []
    if not any((root / d).exists() for d in ['Celeb-real', 'Celeb-synthesis', 'List_of_testing_videos.txt']):
        subs = [p for p in root.iterdir() if p.is_dir()]
        if subs:
            root = subs[0]

    records = []
    list_file = root / 'List_of_testing_videos.txt'
    if list_file.exists():
        for line in list_file.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) < 2: continue
            is_real = int(parts[0])
            vp = root / parts[1]
            if not vp.exists(): continue
            our_label = 0 if is_real == 1 else 1
            records.append({'path': str(vp), 'label': our_label, 'dataset': 'celebdf',
                             'manipulation': 'celebdf_fake' if our_label else 'none',
                             'source_type': 'video'})
    else:
        for folder, lbl in [('Celeb-real', 0), ('YouTube-real', 0), ('Celeb-synthesis', 1)]:
            fp = root / folder
            if fp.exists():
                for vp in sorted(fp.glob('*.mp4')):
                    records.append({'path': str(vp), 'label': lbl, 'dataset': 'celebdf',
                                     'manipulation': 'celebdf_fake' if lbl else 'none',
                                     'source_type': 'video'})
    real = [r for r in records if r['label'] == 0]
    fake = [r for r in records if r['label'] == 1]
    n = min(max_videos // 2, len(real), len(fake))
    if n == 0:
        return []
    recs = random.sample(real, n) + random.sample(fake, n)
    random.shuffle(recs)
    print(f'CelebDF-v2: {len(recs)} (real={n}, fake={n})')
    return recs


# Собираем тренировочные данные
print('=== FaceForensics++ (ahmedelbanby сборник) ===')
train_records = build_ffpp_manifest(CFG['ffpp_root'], CFG['ffpp_max_per_class'])

print('\n=== DFDC Faces (itamargr) ===')
train_records += build_dfdc_faces_manifest(CFG['dfdc_faces'], CFG['max_videos_per_class'])

if len(train_records) < 100:
    print('\n=== DFDC Sample (fallback) ===')
    train_records += build_sample_manifest(CFG['dfdc_sample'])

random.shuffle(train_records)
n_val = max(10, int(len(train_records) * CFG['val_ratio']))
train_val = train_records[:n_val]
train_tr  = train_records[n_val:]
r_cnt = sum(1 for x in train_tr if x['label'] == 0)
f_cnt = sum(1 for x in train_tr if x['label'] == 1)

print('\n=== CelebDF-v2 (cross-dataset) ===')
celebdf_records = build_celebdf_manifest(CFG['celebdf_root'], CFG['celebdf_max'])

out = Path(CFG['output_dir'])
(out / 'train.json').write_text(json.dumps(train_tr, indent=2))
(out / 'val.json').write_text(json.dumps(train_val, indent=2))
(out / 'celebdf_test.json').write_text(json.dumps(celebdf_records, indent=2))
print(f'\nИТОГО: train={len(train_tr)} (real={r_cnt}, fake={f_cnt}), '
      f'val={len(train_val)}, celebdf_test={len(celebdf_records)}')

# Алиасы для совместимости с ячейками init/train
dfdc_train = train_tr
dfdc_val   = train_val


In [ ]:
# ─── 2b. Предварительное декодирование mp4 → JPG + детекция лица ──────────
# КРИТИЧНО: FF++ видео — полнокадровые (лицо = 30-50% кадра, остальное фон).
# DFDC-faces — уже обрезанные лица (не проходят precache, source_type='files').
# Без детекции лица модель видит фон + мелкое лицо → не может учиться.
#
# Решение: Haar cascade для детекции + center crop 65% как fallback.
# Лучшие решения DFDC (selimsef, 1st place) использовали face detector +
# 30% margin от размера лица — мы делаем то же самое.
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

CACHE_ROOT = Path('/kaggle/working/ffpp_cache_v2')  # v2: с детекцией лица
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# ── Детекция лица (Haar cascade — встроен в OpenCV, без pip install) ──
_face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def _crop_face(img_rgb, margin=0.3):
    """Обнаружить крупнейшее лицо, обрезать с margin 30%.
    Fallback: центральный кроп 65% (лицо обычно в центре FF++ видео)."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    faces = _face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=4, minSize=(40, 40))
    h, w = img_rgb.shape[:2]
    if len(faces) > 0:
        # берём самое большое лицо
        areas = [fw * fh for (fx, fy, fw, fh) in faces]
        fx, fy, fw, fh = faces[int(np.argmax(areas))]
        mx, my = int(fw * margin), int(fh * margin)
        x1 = max(0, fx - mx)
        y1 = max(0, fy - my)
        x2 = min(w, fx + fw + mx)
        y2 = min(h, fy + fh + my)
        crop = img_rgb[y1:y2, x1:x2]
        if crop.shape[0] > 10 and crop.shape[1] > 10:
            return crop
    # Fallback: center crop 65%
    cr = 0.65
    ch, cw = int(h * cr), int(w * cr)
    y1, x1 = (h - ch) // 2, (w - cw) // 2
    return img_rgb[y1:y1+ch, x1:x1+cw]


def _decode_one(video_path, out_dir, T=16, sz=224, crop_face=False):
    out_dir = Path(out_dir)
    if out_dir.exists() and len(list(out_dir.glob('f_*.jpg'))) >= T:
        return True  # уже декодирован
    out_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    try:
        with av.open(str(video_path)) as c:
            st = c.streams.video[0]
            total = st.frames or 0
            if   total <= 0: idx = set()
            elif total <= T: idx = set(range(total))
            else:            idx = {int(total/T*k) for k in range(T)}
            i = done = 0
            for pkt in c.demux(st):
                for fr in pkt.decode():
                    if not idx or i in idx:
                        img = fr.to_ndarray(format='rgb24')
                        h, w = img.shape[:2]
                        if max(h, w) > 640:
                            k = 640 / max(h, w)
                            img = cv2.resize(img, (int(w*k), int(h*k)), cv2.INTER_AREA)
                        # Детекция и обрезка лица (для FF++ полнокадровых видео)
                        if crop_face:
                            img = _crop_face(img)
                        img = cv2.resize(img, (sz, sz), cv2.INTER_LINEAR)
                        frames.append(img)
                    i += 1
                    if len(frames) == T: done = 1; break
                if done: break
    except Exception:
        return False
    if not frames:
        return False
    while len(frames) < T:
        frames.append(frames[-1].copy())
    for k, im in enumerate(frames):
        cv2.imwrite(str(out_dir / f'f_{k:03d}.jpg'),
                    cv2.cvtColor(im, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 92])
    return True


def precache_videos(records, cache_root, T, sz, max_workers=4, tag='', crop_face=False):
    """Декодируем все source_type='video' → JPG-папки, переписываем records."""
    vids = [r for r in records if r.get('source_type') == 'video']
    if not vids:
        return records
    jobs = []
    for r in vids:
        vid_id = Path(r['path']).stem
        mani   = r.get('manipulation', 'unk')
        dset   = r.get('dataset', 'x')
        dst    = cache_root / dset / mani / vid_id
        jobs.append((r, dst))
    face_str = ' + face crop' if crop_face else ''
    print(f'{tag}: декодирую {len(jobs)} mp4 → JPG{face_str} (workers={max_workers}) ...')
    ok_cnt = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_decode_one, r['path'], dst, T, sz, crop_face): (r, dst)
                    for r, dst in jobs}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f'decode {tag}'):
            r, dst = futures[fut]
            try:
                ok = fut.result()
            except Exception:
                ok = False
            if ok:
                r['path'] = str(dst)
                r['source_type'] = 'folder'
                ok_cnt += 1
    print(f'{tag}: готово {ok_cnt}/{len(jobs)}')
    return [r for r in records if r.get('source_type') != 'video' or r in vids[:0]]


T, sz = CFG['target_frames'], CFG['image_size']
# crop_face=True: FF++ видео полнокадровые → обрезаем лицо
# DFDC-faces (source_type='files') НЕ проходят через precache → уже face-cropped
dfdc_train = precache_videos(dfdc_train, CACHE_ROOT, T, sz, max_workers=4, tag='train', crop_face=True)
dfdc_val   = precache_videos(dfdc_val,   CACHE_ROOT, T, sz, max_workers=4, tag='val', crop_face=True)

def _count(records):
    d = {'folder':0,'files':0,'video':0}
    for r in records: d[r.get('source_type','video')] = d.get(r.get('source_type','video'),0)+1
    return d
print('После кэша train:', _count(dfdc_train))
print('После кэша val:  ', _count(dfdc_val))
print(f'Размер кэша (MB): {sum(f.stat().st_size for f in CACHE_ROOT.rglob("*.jpg"))/1e6:.0f}')

In [ ]:
# ─── 2c. Загрузка всех кадров в RAM ──────────────────────────────────────
# Вместо записи 12 GB .npy на диск и чтения через mmap/chunks (page cache
# считается в лимит 30 GB на Kaggle и НЕ сбрасывается через fadvise),
# просто держим всё в обычных numpy-массивах.
# Расчёт: 12 GB uint8 (данные) + 3 GB (модель/opt) + 1 GB (overhead) ≈ 16 GB из 30.

def _resize_sq(img, sz):
    h, w = img.shape[:2]
    if max(h, w) > 640:
        k = 640 / max(h, w)
        img = cv2.resize(img, (int(w*k), int(h*k)), cv2.INTER_AREA)
    return cv2.resize(img, (sz, sz), cv2.INTER_LINEAR)


def _load_frames_u8(rec, T, sz):
    """→ (T, H, W, 3) uint8 ndarray."""
    src = rec.get('source_type')
    frames = []
    if src == 'files':
        files = list(rec['files'])
    elif src == 'folder':
        p = Path(rec['path'])
        files = []
        for ext in IMG_EXTS:
            files.extend(p.glob(f'*{ext}'))
        files = sorted(files)
    else:
        return None
    n = len(files)
    if n == 0:
        return None
    idx_list = list(range(n)) if n <= T else [int(n/T*k) for k in range(T)]
    for i in idx_list:
        img = cv2.imread(str(files[i]), cv2.IMREAD_COLOR)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames.append(_resize_sq(img, sz))
    while len(frames) < T:
        frames.append(frames[-1].copy() if frames else np.zeros((sz, sz, 3), np.uint8))
    return np.stack(frames[:T], axis=0).astype(np.uint8)


def preload_to_ram(records, T, sz, tag='train'):
    """Декодируем все кадры → numpy array в RAM (без файлов на диске)."""
    n = len(records)
    frames = np.zeros((n, T, sz, sz, 3), dtype=np.uint8)
    labels = np.zeros(n, dtype=np.int64)
    ok = 0
    for i, rec in enumerate(tqdm(records, desc=f'load {tag}', dynamic_ncols=True)):
        arr = _load_frames_u8(rec, T, sz)
        if arr is not None:
            frames[i] = arr
            ok += 1
        labels[i] = int(rec['label'])
    print(f'{tag}: {ok}/{n} сэмплов, {frames.nbytes/1024**3:.2f} GB в RAM')
    return frames, labels


T_, sz_ = CFG['target_frames'], CFG['image_size']
train_frames, train_labels = preload_to_ram(dfdc_train, T_, sz_, tag='train')
val_frames, val_labels     = preload_to_ram(dfdc_val,   T_, sz_, tag='val')
import gc; gc.collect()
print(f'В RAM: train {train_frames.shape} ({train_frames.nbytes/1024**3:.2f} GB), '
      f'val {val_frames.shape} ({val_frames.nbytes/1024**3:.2f} GB)')

In [ ]:
# ─── 3. Модель EDN-Ad (Linear head + BCE) ─────────────────────────────────
# Архитектура EDN-Ad:
#   Spatial:  EfficientNet-B0 (pretrained) → FPN → SE → projection
#   Temporal: Adaptive Dilated TCN (2 блока)
#   Head:     Dropout → Linear(512, 1) → BCE loss
#
# КРИТИЧНО: ArcFace заменён на Linear head.
# Причина: ArcFace с L2-нормализацией ограничивает оптимизацию на единичной
# сфере, что при random-init FPN/SE/TCN делает обучение невозможным.
# Все топ-решения DFDC (selimsef 1st place, NTech-Lab 3rd) используют
# простой Linear + BCE/CE.

import timm

class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch//r), nn.ReLU(True),
                                 nn.Linear(ch//r, ch), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b,c)).view(b,c,1,1)


class FPN(nn.Module):
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.lat = nn.ModuleList([nn.Conv2d(c,out_ch,1) for c in in_ch])
        self.smo = nn.ModuleList([nn.Conv2d(out_ch,out_ch,3,padding=1) for _ in in_ch])
    def forward(self, feats):
        lats = [l(f) for l,f in zip(self.lat,feats)]
        outs = [lats[-1]]
        for i in reversed(range(len(lats)-1)):
            outs.insert(0, lats[i] + F.interpolate(outs[0], size=lats[i].shape[-2:], mode='nearest'))
        return [s(o) for s,o in zip(self.smo,outs)]


class SpatialEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.bb  = timm.create_model('efficientnet_b0', pretrained=True,
                                      features_only=True, out_indices=(2,3,4))
        self.fpn = FPN(self.bb.feature_info.channels())
        self.se  = nn.ModuleList([SEBlock(128) for _ in range(3)])
        self.pool= nn.AdaptiveAvgPool2d(1)
        self.proj= nn.Sequential(
            nn.Linear(384, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        pyr  = self.fpn(self.bb(x))
        pool = [self.pool(se(p)).flatten(1) for se,p in zip(self.se,pyr)]
        return self.proj(torch.cat(pool, 1))


class AdaptiveDilatedConv1d(nn.Module):
    def __init__(self, ch, k=3, bd=1, md=16):
        super().__init__()
        self.k, self.bd, self.md = k, bd, md
        self.W = nn.Parameter(torch.empty(2*ch, ch, k))
        nn.init.kaiming_uniform_(self.W, a=5**0.5)
        self.b = nn.Parameter(torch.zeros(2*ch))
    def forward(self, x, E):
        phi = 1/(1+E.mean().clamp(1e-6))
        d = int(max(1, min(self.md, round(self.bd*phi.item()))))
        y = F.conv1d(F.pad(x, ((self.k-1)*d, 0)), self.W, self.b, dilation=d)
        a, b = y.chunk(2, 1)
        return a * torch.sigmoid(b)


class TemporalBlock(nn.Module):
    def __init__(self, ch, k, bd, dr):
        super().__init__()
        self.conv = AdaptiveDilatedConv1d(ch, k, bd)
        self.ln = nn.LayerNorm(ch)
        self.drop = nn.Dropout(dr)
    def forward(self, x, E):
        y = self.conv(x, E)
        return self.drop(self.ln(y.transpose(1, 2)).transpose(1, 2)) + x


class TemporalEncoder(nn.Module):
    def __init__(self, ch=512, nl=2, k=3, dr=0.1):
        super().__init__()
        self.blks = nn.ModuleList([TemporalBlock(ch, k, 2**i, dr) for i in range(nl)])
    def forward(self, seq):
        x = seq.transpose(1, 2)
        E = F.pad((x[..., 1:] - x[..., :-1]).pow(2).mean(1, True), (1, 0))
        for blk in self.blks:
            x = blk(x, E)
        return x.mean(-1)


class DeepfakeDetector(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.spatial  = SpatialEncoder(c['embed_dim'])
        self.temporal = TemporalEncoder(c['embed_dim'], c['n_layers'], c['kernel'], c['dropout'])
        self.head = nn.Sequential(
            nn.Dropout(c['dropout']),
            nn.Linear(c['embed_dim'], 1),
        )

    def forward(self, video):
        b, t, ch, h, w = video.shape
        sp  = self.spatial(video.contiguous().reshape(b*t, ch, h, w)).reshape(b, t, -1)
        agg = self.temporal(sp)
        logit = self.head(agg).squeeze(-1)  # (B,)
        return logit

    def freeze_backbone(self):
        """Замораживает EfficientNet — FPN/SE/TCN/head учатся на pretrained features."""
        for p in self.spatial.bb.parameters():
            p.requires_grad = False
        print('  Backbone FROZEN (EfficientNet-B0)')

    def unfreeze_backbone(self):
        """Размораживает EfficientNet для fine-tuning с пониженным LR."""
        for p in self.spatial.bb.parameters():
            p.requires_grad = True
        print('  Backbone UNFROZEN (fine-tuning)')

model_tmp = DeepfakeDetector(CFG)
n_p = sum(p.numel() for p in model_tmp.parameters()) / 1e6
print(f'Model defined ✓ (Linear head + BCE) | {n_p:.1f}M params')
del model_tmp

In [ ]:
# ─── 4. Потери и метрики ───────────────────────────────────────────────────
# BCE loss — стандарт для binary classification в deepfake detection.
# Все топ-решения DFDC используют BCE или CE, НЕ ArcFace.

bce_loss_fn = nn.BCEWithLogitsLoss()

def compute_metrics(labels, probs, prefix=''):
    lb = np.array(labels); pb = np.array(probs)
    if len(set(lb)) < 2:
        return {f'{prefix}auroc':0., f'{prefix}f1':0., f'{prefix}eer':0.}
    auroc = float(roc_auc_score(lb,pb))
    f1    = float(f1_score(lb,(pb>=0.5).astype(int),zero_division=0))
    fpr,tpr,_ = roc_curve(lb,pb)
    fnr = 1-tpr
    eer = float((fpr+fnr)[np.argmin(np.abs(fnr-fpr))]/2)
    return {f'{prefix}auroc':auroc, f'{prefix}f1':f1, f'{prefix}eer':eer}

print('Loss (BCE) & metrics defined ✓')

In [ ]:
# ─── 5. Dataset & препроцессинг ────────────────────────────────────────────
# Аугментация следует лучшим практикам DFDC:
# - Horizontal flip (бесплатно, удваивает разнообразие)
# - Color jitter (brightness/contrast — имитация разных камер)
# - Gaussian blur (имитация low-quality видео)
# - JPEG compression simulation (артефакты сжатия — ключевой сигнал в deepfake)

MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')


def _resize_sq(img, sz):
    h, w = img.shape[:2]
    if max(h, w) > 640:
        k = 640 / max(h, w)
        img = cv2.resize(img, (int(w*k), int(h*k)), cv2.INTER_AREA)
    return cv2.resize(img, (sz, sz), cv2.INTER_LINEAR)


def _augment_frame_u8(img):
    """Аугментация одного кадра (uint8 numpy HWC). Применяется ДО нормализации."""
    # Gaussian blur (p=0.3)
    if random.random() < 0.3:
        ksize = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (ksize, ksize), 0)
    # JPEG compression simulation (p=0.3) — ключевой приём из топ-решений
    if random.random() < 0.3:
        quality = random.randint(30, 80)
        _, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
        img = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    # Brightness/contrast jitter
    if random.random() < 0.5:
        alpha = random.uniform(0.8, 1.2)   # contrast
        beta = random.uniform(-15, 15)      # brightness
        img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return img


class MmapDS(Dataset):
    """Dataset из numpy array в RAM."""
    def __init__(self, frames_arr, labels, augment=False):
        self.frames = frames_arr
        self.labels = labels
        self.aug = augment
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        # frames[idx]: (T, H, W, 3) uint8
        frames = self.frames[idx].copy()  # copy для in-place аугментации

        if self.aug:
            # Покадровая аугментация (blur, JPEG, jitter)
            for i in range(len(frames)):
                frames[i] = _augment_frame_u8(frames[i])

        vid = torch.from_numpy(frames).float()
        vid = vid.permute(0, 3, 1, 2).contiguous().div_(255.0)
        vid.sub_(MEAN).div_(STD)

        if self.aug:
            # Horizontal flip (применяется ко всем кадрам одинаково)
            if random.random() < 0.5:
                vid = vid.flip(-1)

        return {'video': vid, 'label': int(self.labels[idx])}


def load_frames_from_folder(folder, T=16, sz=224):
    p = Path(folder)
    files = []
    for ext in IMG_EXTS:
        files.extend(p.glob(f'*{ext}'))
    files = sorted(files)
    frames = []
    if files:
        n = len(files)
        idx_list = list(range(n)) if n <= T else [int(n/T*k) for k in range(T)]
        for i in idx_list:
            img = cv2.imread(str(files[i]), cv2.IMREAD_COLOR)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            frames.append(_resize_sq(img, sz))
    while len(frames) < T:
        frames.append(frames[-1].copy() if frames else np.zeros((sz, sz, 3), np.uint8))
    arr = torch.from_numpy(np.stack(frames[:T])).float() / 255.
    arr = arr.permute(0, 3, 1, 2)
    return (arr - MEAN) / STD


def load_frames_from_video(path, T=16, sz=224, crop_face=True):
    """Декодирует видео → T кадров с face crop → (T,3,sz,sz).
    КРИТИЧНО: crop_face=True по умолчанию.
    Тренировочные данные face-cropped → eval тоже должен быть face-cropped.
    Без этого cross-dataset eval даёт AUROC ~0.45 (модель видит фон вместо лица)."""
    frames = []
    try:
        with av.open(str(path)) as c:
            st = c.streams.video[0]
            total = st.frames or 0
            if   total <= 0: idx = set()
            elif total <= T: idx = set(range(total))
            else:            idx = {int(total/T*k) for k in range(T)}
            i = done = 0
            for pkt in c.demux(st):
                for fr in pkt.decode():
                    if not idx or i in idx:
                        img = fr.to_ndarray(format='rgb24')
                        h, w = img.shape[:2]
                        if max(h, w) > 640:
                            k = 640 / max(h, w)
                            img = cv2.resize(img, (int(w*k), int(h*k)), cv2.INTER_AREA)
                        if crop_face:
                            img = _crop_face(img)
                        frames.append(cv2.resize(img, (sz, sz), cv2.INTER_LINEAR))
                    i += 1
                    if len(frames) == T: done = 1; break
                if done: break
    except Exception:
        pass
    while len(frames) < T:
        frames.append(frames[-1].copy() if frames else np.zeros((sz, sz, 3), np.uint8))
    arr = torch.from_numpy(np.stack(frames[:T])).float() / 255.
    arr = arr.permute(0, 3, 1, 2)
    return (arr - MEAN) / STD


def load_frames_from_file_list(files, T=16, sz=224):
    files = list(files)
    n = len(files)
    frames = []
    if n:
        idx_list = list(range(n)) if n <= T else [int(n/T*k) for k in range(T)]
        for i in idx_list:
            img = cv2.imread(str(files[i]), cv2.IMREAD_COLOR)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            frames.append(_resize_sq(img, sz))
    while len(frames) < T:
        frames.append(frames[-1].copy() if frames else np.zeros((sz, sz, 3), np.uint8))
    arr = torch.from_numpy(np.stack(frames[:T])).float() / 255.
    arr = arr.permute(0, 3, 1, 2)
    return (arr - MEAN) / STD


def load_frames(rec, T=16, sz=224):
    src = rec.get('source_type')
    if src == 'files':
        return load_frames_from_file_list(rec['files'], T, sz)
    path = rec['path']
    if src == 'folder' or (src is None and Path(path).is_dir()):
        return load_frames_from_folder(path, T, sz)
    return load_frames_from_video(path, T, sz, crop_face=True)


class VideoDS(Dataset):
    def __init__(self, records, T=16, sz=224, augment=False):
        self.recs, self.T, self.sz, self.aug = records, T, sz, augment
    def __len__(self): return len(self.recs)
    def __getitem__(self, idx):
        rec = self.recs[idx]
        vid = load_frames(rec, self.T, self.sz)
        if self.aug:
            if random.random() < .5:
                vid = vid.flip(-1)
        return {'video': vid, 'label': int(rec['label'])}

print('Dataset defined ✓ (with face crop + JPEG/blur/jitter augmentation)')


In [ ]:
# ─── 6. Функции обучения и оценки (BCE) ───────────────────────────────────
import gc
from tqdm.auto import tqdm


def run_epoch(model, loader, opt, scaler, accum, dev,
              train=True, desc='epoch'):
    model.train() if train else model.eval()
    total_loss, n = 0., 0
    all_p, all_l = [], []
    if train and opt is not None:
        opt.zero_grad(set_to_none=True)

    ctx = torch.enable_grad() if train else torch.no_grad()
    pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True, mininterval=1.0)
    with ctx:
        for step, batch in enumerate(pbar):
            v = batch['video'].to(dev, non_blocking=True)
            l = batch['label'].to(dev, non_blocking=True).float()  # BCE needs float

            with torch.amp.autocast('cuda', enabled=CFG['use_amp']):
                logits = model(v)            # (B,) raw logits
                loss = bce_loss_fn(logits, l)
                if train:
                    loss = loss / accum

            if train:
                scaler.scale(loss).backward()
                total_loss += loss.item() * accum
                if (step+1) % accum == 0:
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad'])
                    scaler.step(opt); scaler.update()
                    opt.zero_grad(set_to_none=True)
                pbar.set_postfix(loss=f'{loss.item()*accum:.3f}',
                                  lr=f"{opt.param_groups[0]['lr']:.1e}")
            else:
                total_loss += loss.item()
                probs = torch.sigmoid(logits).cpu().tolist()
                all_p += probs
                all_l += l.cpu().int().tolist()
                pbar.set_postfix(loss=f'{loss.item():.3f}')
            n += 1

            if step % 50 == 49:
                gc.collect()
                _malloc_trim()

        # Flush remaining gradients
        if train and opt is not None and n > 0 and n % accum != 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad'])
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)

    gc.collect()
    torch.cuda.empty_cache()
    _malloc_trim()

    avg_loss = total_loss / max(n, 1)
    if train:
        return avg_loss
    m = compute_metrics(all_l, all_p)
    m['loss'] = avg_loss
    return m


def cross_eval(model, records, dev, tag='CelebDF-v2'):
    print(f'\nCross-dataset evaluation on {tag}...')
    ds = VideoDS(records, T=CFG['target_frames'], sz=CFG['image_size'], augment=False)
    loader = DataLoader(ds, batch_size=CFG['batch']*2,
                        num_workers=0, pin_memory=False)
    m = run_epoch(model, loader, None, None, 1, dev, train=False, desc=tag)
    return m

print('Train/eval functions defined ✓ (BCE loss, sigmoid probs)')

In [ ]:
# ─── 7. Инициализация ─────────────────────────────────────────────────────
torch.backends.cudnn.benchmark = False

train_ds = MmapDS(train_frames, train_labels, augment=True)
val_ds   = MmapDS(val_frames,   val_labels,   augment=False)

train_ld = DataLoader(train_ds, batch_size=CFG['batch'], shuffle=True,
                      num_workers=0, pin_memory=False, drop_last=True)
val_ld   = DataLoader(val_ds,   batch_size=CFG['batch']*2, shuffle=False,
                      num_workers=0, pin_memory=False)

model = DeepfakeDetector(CFG).to(device)
print(f'Single GPU: {torch.cuda.get_device_name(0)}')

# ── Backbone freeze на первые N эпох ──
# Критично: FPN/SE/proj/TCN/head — всё random init.
# Если backbone не заморожен, случайные градиенты от верхних слоёв
# разрушают pretrained features EfficientNet.
# Стратегия: 2 эпохи freeze → верхние слои учатся на хороших features →
# потом unfreeze backbone с маленьким LR.
model.freeze_backbone()

# ── Differential LR ──
# backbone (pretrained) — низкий LR, чтобы не разрушить features
# остальное (random init) — высокий LR для быстрого обучения
backbone_params = list(model.spatial.bb.parameters())
backbone_ids = {id(p) for p in backbone_params}
other_params = [p for p in model.parameters() if id(p) not in backbone_ids]

opt = AdamW([
    {'params': backbone_params, 'lr': CFG['lr_backbone']},   # 2e-5
    {'params': other_params,    'lr': CFG['lr']},             # 1e-3
], weight_decay=CFG['wd'])

sched  = CosineAnnealingLR(opt, T_max=CFG['epochs'])
scaler = torch.amp.GradScaler(enabled=CFG['use_amp'])

n_p = sum(p.numel() for p in model.parameters())/1e6
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
print(f'Model params: {n_p:.1f}M total, {n_trainable:.1f}M trainable (backbone frozen)')
print(f'Train: {len(train_ld)} batches | Val: {len(val_ld)} batches')
print(f'LR: backbone={CFG["lr_backbone"]}, head={CFG["lr"]}')
print(f'Data in RAM ({train_frames.nbytes/1024**3:.1f} GB), no file I/O during training')

In [ ]:
# ─── 8. Цикл обучения ─────────────────────────────────────────────────────
best_auroc = 0.
start_epoch = 0
out        = Path(CFG['output_dir'])
history    = []

# ── Resume from last_model.pt если есть ──
resume_path = out / 'last_model.pt'
if resume_path.exists():
    try:
        ck = torch.load(resume_path, map_location=device)
        model.load_state_dict(ck['state_dict'])
        opt.load_state_dict(ck['opt'])
        scaler.load_state_dict(ck['scaler'])
        sched.load_state_dict(ck['sched'])
        start_epoch = ck['epoch'] + 1
        best_auroc  = ck.get('best_auroc', 0.)
        history     = ck.get('history', [])
        # Восстанавливаем freeze/unfreeze состояние
        if start_epoch >= CFG['freeze_backbone_epochs']:
            model.unfreeze_backbone()
        else:
            model.freeze_backbone()
        print(f'Resumed from epoch {start_epoch}, best AUROC so far: {best_auroc:.4f}')
    except Exception as e:
        print(f'Resume failed, starting fresh: {e}')
        start_epoch = 0

freeze_epochs = CFG['freeze_backbone_epochs']
print(f'Training epochs {start_epoch}..{CFG["epochs"]-1}')
print(f'Phase 1 (ep 0-{freeze_epochs-1}): backbone FROZEN, train FPN/SE/TCN/head only')
print(f'Phase 2 (ep {freeze_epochs}+): backbone UNFROZEN, fine-tune all with differential LR')
SEP = '='*80
print(SEP)

for epoch in tqdm(range(start_epoch, CFG['epochs']), desc='Epochs', position=0):
    gc.collect(); torch.cuda.empty_cache()

    # ── Unfreeze backbone после freeze_epochs ──
    if epoch == freeze_epochs:
        model.unfreeze_backbone()
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
        print(f'  Now training {n_trainable:.1f}M params')

    t0 = time.time()
    tr_loss = run_epoch(model, train_ld, opt, scaler, CFG['grad_accum'],
                        device, train=True, desc=f'train ep{epoch}')
    val_m   = run_epoch(model, val_ld, None, None, 1,
                        device, train=False, desc=f'val ep{epoch}')

    sched.step()
    lr_bb = opt.param_groups[0]['lr']
    lr_head = opt.param_groups[1]['lr']
    dt = time.time()-t0

    row = dict(epoch=epoch, train_loss=round(tr_loss,4),
               val_loss=round(val_m['loss'],4),
               dfdc_auroc=round(val_m['auroc'],4),
               dfdc_f1=round(val_m['f1'],4),
               dfdc_eer=round(val_m['eer'],4),
               lr_bb=round(lr_bb,7), lr_head=round(lr_head,6),
               time_s=round(dt,1))
    history.append(row)

    frozen_str = ' [FROZEN]' if epoch < freeze_epochs else ''
    print(f"Ep{epoch:02d} | tr={tr_loss:.4f} | val={val_m['loss']:.4f} | "
          f"AUROC={val_m['auroc']:.4f} | F1={val_m['f1']:.4f} | "
          f"EER={val_m['eer']:.4f} | lr_h={lr_head:.1e}{frozen_str} | {dt:.0f}s")

    # ── Сохранение чекпоинта ──
    torch.save({
        'epoch':      epoch,
        'state_dict': model.state_dict(),
        'opt':        opt.state_dict(),
        'scaler':     scaler.state_dict(),
        'sched':      sched.state_dict(),
        'best_auroc': best_auroc,
        'history':    history,
        'cfg':        CFG,
    }, out/'last_model.pt')

    if val_m['auroc'] > best_auroc:
        best_auroc = val_m['auroc']
        torch.save({'epoch':epoch,'state_dict':model.state_dict(),
                    'metrics':val_m,'cfg':CFG}, out/'best_model.pt')
        print(f'  ★ New best AUROC {best_auroc:.4f} → best_model.pt')

print(SEP)
print(f'Training done. Best DFDC AUROC: {best_auroc:.4f}')

import pandas as pd
df_hist = pd.DataFrame(history)
df_hist.to_csv(out/'training_history.csv', index=False)
print('History → training_history.csv')

In [ ]:
# ─── 9. Кросс-датасет оценка на CelebDF-v2 ────────────────────────────────
ckpt = torch.load(out/'best_model.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

dfdc_final = run_epoch(model, val_ld, None, None, 1, device,
                       train=False, desc='DFDC final')
print(f"DFDC val (best model): AUROC={dfdc_final['auroc']:.4f} | F1={dfdc_final['f1']:.4f} | EER={dfdc_final['eer']:.4f}")

if celebdf_records:
    celebdf_m = cross_eval(model, celebdf_records, device, 'CelebDF-v2')
    print(f"CelebDF-v2 (cross): AUROC={celebdf_m['auroc']:.4f} | F1={celebdf_m['f1']:.4f} | EER={celebdf_m['eer']:.4f}")
else:
    celebdf_m = {'auroc':None,'f1':None,'eer':None,'loss':None}
    print('WARNING: CelebDF-v2 не загружен, кросс-тест пропущен')

In [ ]:
# ─── 9b. Пересчёт CelebDF-v2 с face crop (hotfix) ─────────────────────────
# Проблема: load_frames_from_video не делал face crop → модель видела фон
# Тренировочные данные были face-cropped, а CelebDF eval — нет → AUROC 0.45
# Фикс: добавляем _crop_face в декодирование видео

def load_frames_from_video_fc(path, T=16, sz=224):
    """Декодирует видео с face crop (как при тренировке)."""
    frames = []
    try:
        with av.open(str(path)) as c:
            st = c.streams.video[0]
            total = st.frames or 0
            if   total <= 0: idx = set()
            elif total <= T: idx = set(range(total))
            else:            idx = {int(total/T*k) for k in range(T)}
            i = done = 0
            for pkt in c.demux(st):
                for fr in pkt.decode():
                    if not idx or i in idx:
                        img = fr.to_ndarray(format='rgb24')
                        h, w = img.shape[:2]
                        if max(h, w) > 640:
                            k = 640 / max(h, w)
                            img = cv2.resize(img, (int(w*k), int(h*k)), cv2.INTER_AREA)
                        img = _crop_face(img)
                        frames.append(cv2.resize(img, (sz, sz), cv2.INTER_LINEAR))
                    i += 1
                    if len(frames) == T: done = 1; break
                if done: break
    except Exception:
        pass
    while len(frames) < T:
        frames.append(frames[-1].copy() if frames else np.zeros((sz, sz, 3), np.uint8))
    arr = torch.from_numpy(np.stack(frames[:T])).float() / 255.
    arr = arr.permute(0, 3, 1, 2)
    return (arr - MEAN) / STD

class VideoDSFaceCrop(Dataset):
    def __init__(self, records, T=16, sz=224):
        self.recs, self.T, self.sz = records, T, sz
    def __len__(self): return len(self.recs)
    def __getitem__(self, idx):
        rec = self.recs[idx]
        vid = load_frames_from_video_fc(rec['path'], self.T, self.sz)
        return {'video': vid, 'label': int(rec['label'])}

# Загружаем лучшую модель
ckpt = torch.load(out/'best_model.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

# DFDC val (без изменений — уже face-cropped)
dfdc_final = run_epoch(model, val_ld, None, None, 1, device,
                       train=False, desc='DFDC final')
print(f"DFDC val: AUROC={dfdc_final['auroc']:.4f} | F1={dfdc_final['f1']:.4f} | EER={dfdc_final['eer']:.4f}")

# CelebDF-v2 с face crop
if celebdf_records:
    print('\nCelebDF-v2 с face crop...')
    ds_fc = VideoDSFaceCrop(celebdf_records, T=CFG['target_frames'], sz=CFG['image_size'])
    ld_fc = DataLoader(ds_fc, batch_size=CFG['batch']*2, num_workers=0, pin_memory=False)
    celebdf_m = run_epoch(model, ld_fc, None, None, 1, device, train=False, desc='CelebDF-v2 FC')
    print(f"CelebDF-v2 (face crop): AUROC={celebdf_m['auroc']:.4f} | F1={celebdf_m['f1']:.4f} | EER={celebdf_m['eer']:.4f}")
else:
    celebdf_m = {'auroc': None, 'f1': None, 'eer': None}

print('\n' + '='*55)
print('  ИТОГОВЫЕ РЕЗУЛЬТАТЫ (EDN-Ad) — с face crop fix')
print('='*55)
results = pd.DataFrame([
    {'Датасет': 'DFDC (val)', 'AUROC': round(dfdc_final['auroc'],4),
     'F1': round(dfdc_final['f1'],4), 'EER': round(dfdc_final['eer'],4)},
    {'Датасет': 'CelebDF-v2 (cross)', 'AUROC': round(celebdf_m['auroc'],4) if celebdf_m['auroc'] else 'N/A',
     'F1': round(celebdf_m['f1'],4) if celebdf_m['f1'] else 'N/A',
     'EER': round(celebdf_m['eer'],4) if celebdf_m['eer'] else 'N/A'},
])
print(results.to_string(index=False))
print('='*55)

In [ ]:
# ─── 10. Итоговая таблица результатов ─────────────────────────────────────
import matplotlib.pyplot as plt

# ── Таблица для диплома ──
results = pd.DataFrame([
    {'Датасет': 'DFDC (val, in-domain)',
     'AUROC':   round(dfdc_final['auroc'],4),
     'F1':      round(dfdc_final['f1'],4),
     'EER':     round(dfdc_final['eer'],4)},
    {'Датасет': 'CelebDF-v2 (cross-dataset)',
     'AUROC':   round(celebdf_m['auroc'],4) if celebdf_m['auroc'] else 'N/A',
     'F1':      round(celebdf_m['f1'],4)    if celebdf_m['f1']    else 'N/A',
     'EER':     round(celebdf_m['eer'],4)   if celebdf_m['eer']   else 'N/A'},
])

print('\n' + '='*55)
print('  ИТОГОВЫЕ РЕЗУЛЬТАТЫ  (EDN-Ad)')
print('='*55)
print(results.to_string(index=False))
print('='*55)
results.to_csv(out/'final_results.csv', index=False)
print('Таблица сохранена → final_results.csv')

# ── Графики ──
fig, axes = plt.subplots(1,3,figsize=(16,4))
fig.suptitle('EDN-Ad Training on DFDC', fontsize=13)

axes[0].plot(df_hist['epoch'], df_hist['train_loss'], label='Train')
axes[0].plot(df_hist['epoch'], df_hist['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(df_hist['epoch'], df_hist['dfdc_auroc'], 'g-o', label='DFDC val')
if celebdf_m['auroc']:
    axes[1].axhline(celebdf_m['auroc'], color='orange', ls='--', label=f'CelebDF-v2 cross ({celebdf_m["auroc"]:.4f})')
axes[1].set_title('AUROC'); axes[1].set_ylim(0,1); axes[1].legend(); axes[1].grid(True)

axes[2].plot(df_hist['epoch'], df_hist['dfdc_f1'],  'b-', label='F1')
axes[2].plot(df_hist['epoch'], df_hist['dfdc_eer'], 'r-', label='EER')
axes[2].set_title('F1 & EER (DFDC val)'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(out/'results.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён → results.png')

print('\n📥 Скачай из Output:')
print('  best_model.pt       — веса модели')
print('  final_results.csv   — таблица результатов')
print('  results.png         — графики обучения')
print('  training_history.csv— история по эпохам')